---

# 📦 Section 4: Building Reproducible Model Packages

⏱️ **Time:** 25 minutes

### The Reproducibility Challenge

**Real-World Scenario:**

Six months after deploying your model, you need to:
1. Reproduce the exact same model (regulatory audit)
2. Retrain with new data (same architecture)
3. Debug a prediction discrepancy

**You try to load your model:**
```python
model = joblib.load('model.pkl')
```

**Error:**
```
ValueError: numpy.dtype size changed
ModuleNotFoundError: No module named 'utils'
AttributeError: 'RandomForestClassifier' has no attribute 'n_features_in_'
```

**Why?**
- numpy version changed (1.24 → 1.26)
- sklearn version changed (1.3 → 1.5)
- Custom utils.py file missing
- Python version changed (3.9 → 3.11)

**This is the REPRODUCIBILITY problem!** 😱

---

**The Solution:** Build complete, self-contained, reproducible packages

## 4.1: Environment Specifications

### Why Environment Specs Matter

Your model depends on:
- **Python version** (3.9 vs 3.11 = different behavior)
- **Library versions** (sklearn 1.3 vs 1.5 = breaking changes)
- **System dependencies** (C++ libraries, etc.)
- **Hardware** (CPU vs GPU can affect results)

**Without exact environment specs, you CANNOT reproduce your model!**

---

### Two Approaches: requirements.txt vs conda.yaml

**requirements.txt (Lightweight)**
```txt
scikit-learn==1.3.0
numpy==1.24.3
pandas==2.0.3
mlflow==2.9.2
```

**Pros:**
- ✅ Simple
- ✅ Fast to create
- ✅ pip-compatible

**Cons:**
- ❌ Python-only packages
- ❌ No Python version specified
- ❌ No system dependencies

**conda.yaml (Complete)**
```yaml
name: model_v1_2_0
channels:
  - defaults
dependencies:
  - python=3.11.5
  - pip
  - pip:
    - scikit-learn==1.3.0
    - numpy==1.24.3
    - pandas==2.0.3
    - mlflow==2.9.2
```

**Pros:**
- ✅ Specifies Python version
- ✅ Handles system dependencies
- ✅ Complete environment
- ✅ Cross-platform

**Cons:**
- ❌ Requires conda
- ❌ Larger files

---

### 🎯 Best Practice: Include BOTH!

**Why both?**
- requirements.txt: Quick pip install for development
- conda.yaml: Complete environment for production

**Pro Tip:** Always pin EXACT versions!

In [3]:
# Hands-On: Generating Environment Specifications

import sys
import subprocess

print("📦 Generating Environment Specifications\n")
print("="*70)

# === METHOD 1: requirements.txt ===
print("\n1️⃣ Creating requirements.txt...\n")

# Get installed packages
try:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'freeze'],
        capture_output=True,
        text=True
    )
    all_packages = result.stdout
    
    # Filter to ML-specific packages
    ml_packages = []
    important_packages = [
        'scikit-learn', 'sklearn', 'numpy', 'pandas', 'mlflow',
        'xgboost', 'lightgbm', 'matplotlib', 'seaborn', 'joblib'
    ]
    
    for line in all_packages.split('\n'):
        if any(pkg in line.lower() for pkg in important_packages):
            ml_packages.append(line)
    
    # Create requirements.txt
    requirements_content = "\n".join(ml_packages)
    
    with open('requirements.txt', 'w') as f:
        f.write(requirements_content)
    
    print("✅ requirements.txt created:")
    print("   File: requirements.txt")
    print("   Content:")
    for pkg in ml_packages:
        print(f"      {pkg}")
    
except Exception as e:
    print(f"⚠️  Could not generate requirements.txt: {e}")

# === METHOD 2: conda.yaml ===
print("\n" + "="*70)
print("\n2️⃣ Creating conda.yaml...\n")

conda_yaml = f"""
name: model_v1_2_0
channels:
  - defaults
  - conda-forge
dependencies:
  - python={sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}
  - pip
  - pip:
    - scikit-learn==1.3.0
    - numpy==1.24.3
    - pandas==2.0.3
    - mlflow==2.9.2
    - joblib==1.3.2
    - xgboost==2.0.3
    - matplotlib==3.7.2
""".strip()

with open('conda.yaml', 'w') as f:
    f.write(conda_yaml)

print("✅ conda.yaml created:")
print("   File: conda.yaml")
print("   Content:")
print(conda_yaml)

print("\n" + "="*70)
print("\n💡 USAGE:")
print("   pip install -r requirements.txt")
print("   conda env create -f conda.yaml")
print("\n🎯 Always pin EXACT versions for reproducibility!")

📦 Generating Environment Specifications


1️⃣ Creating requirements.txt...

✅ requirements.txt created:
   File: requirements.txt
   Content:
      joblib==1.5.2
      lightgbm==4.6.0
      matplotlib==3.10.7
      matplotlib-inline==0.2.1
      mlflow==3.6.0
      mlflow-skinny==3.6.0
      mlflow-tracing==3.6.0
      numpy==2.3.4
      pandas==2.3.3
      scikit-learn==1.7.2
      seaborn==0.13.2
      xgboost==3.1.1


2️⃣ Creating conda.yaml...

✅ conda.yaml created:
   File: conda.yaml
   Content:
name: model_v1_2_0
channels:
  - defaults
  - conda-forge
dependencies:
  - python=3.11.9
  - pip
  - pip:
    - scikit-learn==1.3.0
    - numpy==1.24.3
    - pandas==2.0.3
    - mlflow==2.9.2
    - joblib==1.3.2
    - xgboost==2.0.3
    - matplotlib==3.7.2


💡 USAGE:
   pip install -r requirements.txt
   conda env create -f conda.yaml

🎯 Always pin EXACT versions for reproducibility!


## 4.2: Model Signatures & Schemas

### Why Schemas Matter

**Real-World Disaster:**

Production API receives this input:
```python
{
    "age": "25",           # String instead of int!
    "tenure": None,        # Missing value!
    "monthly_charges": -50, # Negative price!
    "total_charges": "N/A" # Invalid type!
}
```

Without validation:
- Model crashes
- Or worse: Returns garbage predictions!
- Production is down
- Customers affected

**Solution: Input/Output Schemas!**

---

### MLflow Model Signatures

**What is a signature?**
- Specifies input schema (features, types, shapes)
- Specifies output schema (prediction format)
- Automatic validation on inference
- Documentation for API consumers

**Benefits:**
- ✅ Type safety (catch errors early)
- ✅ Clear API contract
- ✅ Auto-documentation
- ✅ Validation on every prediction
- ✅ Better error messages

**Example Signature:**
```python
from mlflow.models import infer_signature

# Infer from data
signature = infer_signature(X_train, predictions)

# Save model with signature
mlflow.sklearn.log_model(model, "model", signature=signature)
```

In [4]:
# Hands-On: Creating Model Signatures

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import mlflow
from mlflow.models import infer_signature
import json

print("📐 Creating Model Signatures\n")
print("="*70)

# Train a simple model
print("\n1️⃣ Training a sample model...")
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X_train, y_train)
print("   ✅ Model trained")

# Make predictions
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)

# === METHOD 1: Infer Signature Automatically ===
print("\n2️⃣ Inferring signature automatically...")
signature = infer_signature(X_train, predictions)
print("   ✅ Signature inferred")
print(f"\n   Input schema: {signature.inputs}")
print(f"   Output schema: {signature.outputs}")

# === METHOD 2: Create Signature Manually ===
print("\n" + "="*70)
print("\n3️⃣ Creating signature manually for more control...")

from mlflow.types.schema import Schema, ColSpec
from mlflow.models.signature import ModelSignature

# Define input schema
input_schema = Schema([
    ColSpec("double", "sepal length (cm)"),
    ColSpec("double", "sepal width (cm)"),
    ColSpec("double", "petal length (cm)"),
    ColSpec("double", "petal width (cm)"),
])

# Define output schema
output_schema = Schema([ColSpec("long")])

# Create signature
manual_signature = ModelSignature(
    inputs=input_schema,
    outputs=output_schema
)

print("   ✅ Manual signature created")
print(f"\n   Input schema: {manual_signature.inputs}")
print(f"   Output schema: {manual_signature.outputs}")

# === Save Signature to JSON ===
print("\n" + "="*70)
print("\n4️⃣ Saving signature to JSON file...")

signature_dict = {
    "inputs": [
        {"name": col, "type": "double"}
        for col in X_train.columns
    ],
    "outputs": [
        {"type": "long", "description": "Predicted class label"}
    ],
    "example_input": X_test.iloc[0].to_dict(),
    "example_output": int(predictions[0]),
}

with open('signature.json', 'w') as f:
    json.dump(signature_dict, f, indent=2)

print("   ✅ Signature saved to: signature.json")
print("\n   Content:")
print(json.dumps(signature_dict, indent=2))

print("\n" + "="*70)
print("\n💡 WHY SIGNATURES MATTER:")
print("   • Type validation prevents crashes")
print("   • Clear API documentation")
print("   • Better error messages")
print("   • Catches data issues early")
print("\n🎯 Always include signatures in production models!")

📐 Creating Model Signatures


1️⃣ Training a sample model...
   ✅ Model trained

2️⃣ Inferring signature automatically...
   ✅ Signature inferred

   Input schema: ['sepal length (cm)': double (required), 'sepal width (cm)': double (required), 'petal length (cm)': double (required), 'petal width (cm)': double (required)]
   Output schema: [Tensor('int64', (-1,))]


3️⃣ Creating signature manually for more control...
   ✅ Manual signature created

   Input schema: ['sepal length (cm)': double (required), 'sepal width (cm)': double (required), 'petal length (cm)': double (required), 'petal width (cm)': double (required)]
   Output schema: [long (required)]


4️⃣ Saving signature to JSON file...
   ✅ Signature saved to: signature.json

   Content:
{
  "inputs": [
    {
      "name": "sepal length (cm)",
      "type": "double"
    },
    {
      "name": "sepal width (cm)",
      "type": "double"
    },
    {
      "name": "petal length (cm)",
      "type": "double"
    },
    {
      "name

## 4.3: Random Seed Management

### The Reproducibility Mystery

**Scenario:**
```python
# Training Day 1
model = RandomForestClassifier()
model.fit(X_train, y_train)
# Accuracy: 0.8789
```

**Six months later:**
```python
# Trying to reproduce
model = RandomForestClassifier()
model.fit(X_train, y_train)  # Same data!
# Accuracy: 0.8654  # DIFFERENT!!! 😱
```

**Why?** Missing random seed!

---

### Sources of Randomness in ML

1. **Python's random module**
   - Used by: Data shuffling, splits
   
2. **NumPy's random**
   - Used by: sklearn, feature engineering
   
3. **Model-specific seeds**
   - RandomForest: `random_state`
   - XGBoost: `random_seed`
   - Neural networks: Multiple seeds!
   
4. **Framework-specific (PyTorch/TensorFlow)**
   - torch.manual_seed()
   - tf.random.set_seed()
   
5. **CUDA (GPU) operations**
   - Non-deterministic by default!
   - Need special flags for reproducibility

**You must control ALL of these!**

---

### Complete Seed Management

In [5]:
# Hands-On: Complete Reproducibility Setup

import random
import numpy as np
import os

print("🌱 Setting Up Complete Reproducibility\n")
print("="*70)

def set_all_seeds(seed=42):
    """
    Set all random seeds for complete reproducibility.
    
    This is a PRODUCTION-GRADE reproducibility function!
    Use this at the start of EVERY training script.
    """
    print(f"\n🌱 Setting all random seeds to: {seed}")
    
    # 1. Python's random module
    random.seed(seed)
    print(f"   ✅ Python random seed set")
    
    # 2. NumPy
    np.random.seed(seed)
    print(f"   ✅ NumPy random seed set")
    
    # 3. Environment variables (affects some libraries)
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"   ✅ PYTHONHASHSEED set")
    
    # 4. PyTorch (if available)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # For multi-GPU
        
        # Make CUDA operations deterministic
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        print(f"   ✅ PyTorch seeds set (CPU + CUDA)")
        print(f"   ✅ CUDA deterministic mode enabled")
    except ImportError:
        print(f"   ℹ️  PyTorch not available (skipping)")
    
    # 5. TensorFlow (if available)
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
        print(f"   ✅ TensorFlow seed set")
    except ImportError:
        print(f"   ℹ️  TensorFlow not available (skipping)")
    
    print(f"\n   🎯 All available seeds configured!")
    return seed

# Use it!
RANDOM_SEED = 42
set_all_seeds(RANDOM_SEED)

print("\n" + "="*70)

# Demonstrate reproducibility
print("\n📊 Demonstrating Reproducibility\n")

from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=RANDOM_SEED
)

# Train 3 times with same seed
results = []
for i in range(3):
    set_all_seeds(RANDOM_SEED)  # Reset seeds
    model = RandomForestClassifier(n_estimators=10, random_state=RANDOM_SEED)
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    results.append(acc)
    print(f"   Run {i+1}: Accuracy = {acc:.6f}")

# Check if all identical
all_same = len(set(results)) == 1
print(f"\n   ✅ All results identical: {all_same}")

if all_same:
    print("   🎉 Perfect reproducibility achieved!")
else:
    print("   ⚠️  Results differ - check seed management")

print("\n" + "="*70)
print("\n💡 REPRODUCIBILITY CHECKLIST:")
print("   ✅ Set Python random seed")
print("   ✅ Set NumPy random seed")
print("   ✅ Set model random_state")
print("   ✅ Set PYTHONHASHSEED")
print("   ✅ Set PyTorch seeds (if using PyTorch)")
print("   ✅ Enable CUDA determinism (if using GPU)")
print("   ✅ Set TensorFlow seed (if using TF)")
print("   ✅ Save seed in model metadata")
print("\n🎯 Call set_all_seeds() at the START of every training script!")

🌱 Setting Up Complete Reproducibility


🌱 Setting all random seeds to: 42
   ✅ Python random seed set
   ✅ NumPy random seed set
   ✅ PYTHONHASHSEED set
   ✅ PyTorch seeds set (CPU + CUDA)
   ✅ CUDA deterministic mode enabled
   ✅ TensorFlow seed set

   🎯 All available seeds configured!


📊 Demonstrating Reproducibility


🌱 Setting all random seeds to: 42
   ✅ Python random seed set
   ✅ NumPy random seed set
   ✅ PYTHONHASHSEED set
   ✅ PyTorch seeds set (CPU + CUDA)
   ✅ CUDA deterministic mode enabled
   ✅ TensorFlow seed set

   🎯 All available seeds configured!
   Run 1: Accuracy = 1.000000

🌱 Setting all random seeds to: 42
   ✅ Python random seed set
   ✅ NumPy random seed set
   ✅ PYTHONHASHSEED set
   ✅ PyTorch seeds set (CPU + CUDA)
   ✅ CUDA deterministic mode enabled
   ✅ TensorFlow seed set

   🎯 All available seeds configured!
   Run 2: Accuracy = 1.000000

🌱 Setting all random seeds to: 42
   ✅ Python random seed set
   ✅ NumPy random seed set
   ✅ PYTHONHASHSEED set
  

### 📋 Complete Reproducibility Template

**Every training script should start with this:**

```python
import random
import numpy as np
import os

# === REPRODUCIBILITY SETUP ===
RANDOM_SEED = 42  # Document this!

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    # Add PyTorch/TF if needed

set_all_seeds(RANDOM_SEED)

# === TRAINING ===
model = RandomForestClassifier(random_state=RANDOM_SEED)
model.fit(X_train, y_train)

# === SAVE WITH SEED ===
metadata = {
    "random_seed": RANDOM_SEED,
    # ... other metadata
}
```

**Pro Tips:**
- Document seed value clearly
- Save seed in model metadata
- Use same seed for train/val/test splits
- Test reproducibility regularly
- Be aware GPU operations may still vary slightly